# Incidenti stradali in Italia — dai 7.000 morti del 2001 al plateau del 2018

Dataset: `mit_incidentalita_mensile` (MIT — Ministero delle Infrastrutture e dei Trasporti)

Analisi pubblica: [README](../README.md)

Serie mensile di incidenti, morti, feriti e indicatori di mortalita', gravita' e lesivita' lungo la rete stradale italiana. 216 mesi, 2001-2018.
Il notebook valida i dati e genera le figure usate nel README.

In [1]:
import duckdb
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("SET s3_region='us-east-1';")
con.execute("SET s3_access_key_id='';")
con.execute("SET s3_secret_access_key='';")
con.execute("SET s3_session_token='';")

GCS_PATH = 'gs://dataciviclab-clean/mit_incidentalita_mensile/2001/mit_incidentalita_mensile_2001_clean.parquet'

# Verifica copertura
cov = con.execute(f'''SELECT COUNT(*) AS n, COUNT(DISTINCT anno) AS anni, MIN(anno) AS da, MAX(anno) AS a FROM '{GCS_PATH}' ''').fetchone()
print('Copertura:', cov)

Copertura: (216, 18, 2001, 2018)


In [2]:
# 1. Trend annuale — morti, feriti e incidenti per anno

df_trend = con.execute(f'''
    SELECT anno,
           SUM(morti) AS morti,
           SUM(feriti) AS feriti,
           SUM(incidenti) AS incidenti
    FROM '{GCS_PATH}'
    GROUP BY anno
    ORDER BY anno
''').fetchdf()

df_trend['var_morti'] = df_trend['morti'].pct_change() * 100
print('Trend annuale (morti, feriti, incidenti, var % morti):')
print(df_trend.round(1).to_string(index=False))

Trend annuale (morti, feriti, incidenti, var % morti):
 anno  morti   feriti  incidenti  var_morti
 2001 7096.0 293384.0   244272.0        NaN
 2002 6980.0 378492.0   265402.0       -1.6
 2003 6563.0 330879.0   252271.0       -6.0
 2004 6122.0 318141.0   227074.0       -6.7
 2005 5818.0 334858.0   240011.0       -5.0
 2006 5669.0 308448.0   221816.0       -2.6
 2007 5131.0 325850.0   230871.0       -9.5
 2008 4731.0 289373.0   218963.0       -7.8
 2009 4237.0 307258.0   200096.0      -10.4
 2010 4114.0 256795.0   212997.0       -2.9
 2011 3860.0 271967.0   205638.0       -6.2
 2012 3753.0 203531.0   188228.0       -2.8
 2013 3385.0 235443.0   153741.0       -9.8
 2014 3381.0 251147.0   177031.0       -0.1
 2015 3428.0 246920.0   174539.0        1.4
 2016 3283.0 249175.0   175791.0       -4.2
 2017 3378.0 246750.0   174933.0        2.9
 2018 3334.0 242919.0   172553.0       -1.3


In [3]:
# Figura 1: Trend annuale — morti (barre) e incidenti (linea)

fig, ax = plt.subplots(figsize=(12, 5))

bars = ax.bar(df_trend['anno'].astype(str), df_trend['morti'], color='#95a5a6', edgecolor='white', width=0.7)
bars[0].set_color('#2c3e50')
bars[-1].set_color('#e74c3c')

ax.set_ylabel('Morti (anno)')
ax.set_title('Morti stradali in Italia, 2001-2018')

ax2 = ax.twinx()
ax2.plot(df_trend['anno'].astype(str), df_trend['incidenti'], color='#27ae60', marker='o', linewidth=2, label='Incidenti')
ax2.set_ylabel('Incidenti (anno)')

for bar, val in zip(bars, df_trend['morti']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100, f'{int(val)}',
            ha='center', va='bottom', fontsize=8)

ax.legend([bars], ['Morti'], loc='upper left')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.savefig('../figures/mit_incidentalita_trend_annuale.png', dpi=150, bbox_inches='tight')
print('saved trend_annuale')

saved trend_annuale


In [4]:
# 2. Il plateau — morti stradali dal 2013 al 2018

df_p = df_trend[df_trend['anno'] >= 2013].copy()
print('Morti 2013-2018:')
print(df_p[['anno', 'morti', 'var_morti']].round(2).to_string(index=False))

delta = df_p['morti'].iloc[-1] - df_p['morti'].iloc[0]
print(f'Variazione assoluta 2013->2018: {int(delta):+d} morti')

Morti 2013-2018:
 anno  morti  var_morti
 2013 3385.0      -9.81
 2014 3381.0      -0.12
 2015 3428.0       1.39
 2016 3283.0      -4.23
 2017 3378.0       2.89
 2018 3334.0      -1.30
Variazione assoluta 2013->2018: -51 morti


In [5]:
# Figura 2: Il plateau degli ultimi anni

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_p['anno'], df_p['morti'], color='#e74c3c', marker='o', linewidth=2)
ax.set_ylabel('Morti')
ax.set_title('Morti stradali 2013-2018: la discesa si ferma')
ax.set_xlim(2012.5, 2018.5)
for x, y in zip(df_p['anno'], df_p['morti']):
    ax.annotate(f'{int(y)}', (x, y), textcoords='offset points', xytext=(0, 8), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/mit_incidentalita_plateau.png', dpi=150, bbox_inches='tight')
print('saved plateau')

saved plateau


In [6]:
# 3. Stagionalita' — media mensile di morti e indice di mortalita'

df_mese = con.execute(f'''
    SELECT mese,
           ROUND(avg(morti), 1) AS avg_morti,
           ROUND(SUM(morti) / SUM(incidenti) * 100, 2) AS indice_mortalita
    FROM '{GCS_PATH}'
    GROUP BY mese
    ORDER BY MIN(mese_numero)
''').fetchdf()
print("Stagionalita' (media morti/mese e indice di mortalita'):")
print(df_mese[['mese', 'avg_morti', 'indice_mortalita']].to_string(index=False))

Stagionalita' (media morti/mese e indice di mortalita'):
     mese  avg_morti  indice_mortalita
  Gennaio      343.1              2.21
 Febbraio      298.3              2.06
    Marzo      347.9              2.32
   Aprile      363.0              2.16
   Maggio      408.0              2.14
   Giugno      445.2              2.21
   Luglio      493.5              2.38
   Agosto      445.5              3.03
Settembre      396.7              2.18
  Ottobre      394.4              2.10
 Novembre      362.2              2.07
 Dicembre      383.3              2.31


In [7]:
# Figura 3: Stagionalita' — morti medi per mese e indice di mortalita' (agosto in evidenza)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(df_mese['mese'], df_mese['avg_morti'], color='#95a5a6', edgecolor='white', width=0.6)
bars[6].set_color('#e74c3c')   # Luglio
bars[7].set_color('#c0392b')   # Agosto
ax.set_ylabel('Morti medi (mese)')
ax.set_title('Morti stradali per mese — media 2001-2018')

ax2 = ax.twinx()
ax2.plot(df_mese['mese'], df_mese['indice_mortalita'], color='#27ae60', marker='o', linewidth=2, label='Morti ogni 100 incidenti')
ax2.set_ylabel("Indice di mortalita'")
ax2.set_ylim(0, ax2.get_ylim()[1] * 1.4)

ax.legend([bars], ['Morti medi'], loc='upper left')
ax2.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig('../figures/mit_incidentalita_stagionalita.png', dpi=150, bbox_inches='tight')
print('saved stagionalita')

saved stagionalita


In [8]:
# 4. Indice di mortalita' — morti ogni 100 incidenti, per anno

df_ind = con.execute(f'''
    SELECT anno,
           ROUND(SUM(morti) / SUM(incidenti) * 100, 2) AS indice_mortalita
    FROM '{GCS_PATH}'
    GROUP BY anno
    ORDER BY anno
''').fetchdf()
print("Indice di mortalita' per anno:")
print(df_ind.to_string(index=False))

Indice di mortalita' per anno:
 anno  indice_mortalita
 2001              2.90
 2002              2.63
 2003              2.60
 2004              2.70
 2005              2.42
 2006              2.56
 2007              2.22
 2008              2.16
 2009              2.12
 2010              1.93
 2011              1.88
 2012              1.99
 2013              2.20
 2014              1.91
 2015              1.96
 2016              1.87
 2017              1.93
 2018              1.93


In [9]:
# Figura 4: Indice di mortalita' stradale — trend 2001-2018

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(df_ind['anno'], df_ind['indice_mortalita'], alpha=0.15, color='#2c3e50')
ax.plot(df_ind['anno'], df_ind['indice_mortalita'], color='#2c3e50', marker='o', linewidth=2)
ax.set_ylabel('Morti ogni 100 incidenti')
ax.set_title("Quanto e' letale la strada — indice di mortalita' 2001-2018")
ax.set_xlim(2000.8, 2018.2)
ax.annotate(f"{df_ind['indice_mortalita'].iloc[0]:.2f}", (df_ind['anno'].iloc[0], df_ind['indice_mortalita'].iloc[0]),
            textcoords='offset points', xytext=(-28, 4), color='#2c3e50', fontweight='bold')
ax.annotate(f"{df_ind['indice_mortalita'].iloc[-1]:.2f}", (df_ind['anno'].iloc[-1], df_ind['indice_mortalita'].iloc[-1]),
            textcoords='offset points', xytext=(-12, 8), color='#e74c3c', fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/mit_incidentalita_indice_mortalita.png', dpi=150, bbox_inches='tight')
print('saved indice_mortalita')

saved indice_mortalita


In [10]:
# Riepilogo — numeri chiave del periodo 2001-2018

import glob
tot = con.execute(f"SELECT SUM(morti), SUM(feriti), SUM(incidenti) FROM '{GCS_PATH}'").fetchone()
morti_tot, feriti_tot, incidenti_tot = tot
m1 = int(df_trend['morti'].iloc[0])
m2 = int(df_trend['morti'].iloc[-1])
print('RIEPILOGO — Incidenti stradali Italia 2001-2018')
print(f'Morti totali nel periodo: {morti_tot:,}')
print(f'Feriti totali: {feriti_tot:,}')
print(f'Incidenti totali: {incidenti_tot:,}')
print(f'Morti 2001: {m1:,}  ->  2018: {m2:,}  (var. {-round((m1 - m2) / m1 * 100, 1)}%)')
print()
print('Figure generate:')
for f in sorted(glob.glob('../figures/mit_incidentalita_*.png')):
    print(f'  OK {f}')

RIEPILOGO — Incidenti stradali Italia 2001-2018
Morti totali nel periodo: 84,263
Feriti totali: 5,091,330
Incidenti totali: 3,736,227
Morti 2001: 7,096  ->  2018: 3,334  (var. -53.0%)

Figure generate:
  OK ../figures/mit_incidentalita_indice_mortalita.png
  OK ../figures/mit_incidentalita_plateau.png
  OK ../figures/mit_incidentalita_stagionalita.png
  OK ../figures/mit_incidentalita_trend_annuale.png
